# Bot Detection on Adobe Experience Platform — User-Agent based

Interactive companion to `src/bot_detection.py`: same logic, one cell at a time, so you can inspect
results before uploading anything. The upload cell is the very last one, deliberately separated: you
can run everything above it with zero risk of touching anything remote.

**Before running this against your own AEP instance**, make sure you've edited the configuration
section at the top of `src/bot_detection.py` (`XDM_NAMESPACE`, `DATASETS`, `BOT_TABLE`) — see the
project README for details. This notebook imports that configuration directly, so editing the script
is enough; you don't need to duplicate anything here.

Dependencies:
```
pip install -r ../requirements.txt
```

## 1. Imports and configuration

In [ ]:
import os
import sys
import getpass

import pandas as pd
import psycopg2
from psycopg2 import sql

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# Make src/ importable regardless of where Jupyter was launched from
# (skip this if you ran `pip install -e .` from the repo root).
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

# Core detection engine (no AEP dependency) + the AEP-specific integration layer.
from bot_detection import load_detectors, load_bot_categorizer, classify, classify_parallel
from bot_detection.aep_pipeline import (
    XDM_NAMESPACE, DATASETS, DATASET_SUFFIX, DEFAULT_BOT_TABLE, EXTRACTION_LIMIT,
    build_union_query, resolve_bot_table, AepConfig, SftpConfig,
)

try:
    from dotenv import find_dotenv, load_dotenv
    env_path = find_dotenv(usecwd=True)
    if env_path:
        load_dotenv(env_path)
        print(f"Loaded variables from {env_path}")
    else:
        print("No .env file found: falling back to getpass prompts below.")
except ImportError:
    print("python-dotenv not installed: falling back to getpass prompts below.")

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **kw: x

In [ ]:
AEP = {
    "host": os.getenv("AEP_HOST") or input("AEP host: "),
    "port": int(os.getenv("AEP_PORT", "80")),
    "dbname": os.getenv("AEP_DBNAME", "prod:all"),
    "user": os.getenv("AEP_USER") or input("AEP user: "),
    "password": os.getenv("AEP_PASSWORD") or getpass.getpass("AEP password: "),
    "sslmode": "require",
}
print("AEP config ready:", AEP["host"])

In [ ]:
# Opened once, reused by every cell below.
conn = psycopg2.connect(**AEP)
print("Connected.")

## 2. Parameters

`XDM_NAMESPACE`, `DATASETS`, `BOT_TABLE` come from `src/bot_detection.py` (see cell above) — edit
them there, not here, so the script and this notebook never drift apart.

In [ ]:
DAYS = 20
MIN_VOTES = 1
MAX_EXPECTED_ROWS = 50_000

print(f"Namespace: {XDM_NAMESPACE} | datasets: {list(DATASETS)} | bot table: {BOT_TABLE}")
print(f"Window: {DAYS} days | vote threshold: {MIN_VOTES}")

### Resolve the bot table

If `BOT_TABLE` has since been renamed in AEP (e.g. it's recreated periodically under a timestamped
name in your environment), this looks for it by prefix instead of failing outright.

In [ ]:
BOT_TABLE = resolve_bot_table(conn, expected=DEFAULT_BOT_TABLE)
print("Using bot table:", BOT_TABLE)

## 3. Detectors

In [ ]:
detectors = load_detectors()
categorizer = load_bot_categorizer()
print("Active detectors:", ", ".join(name for name, _ in detectors))

### Smoke test

Verify the ensemble behaves as expected on known cases before spending a real query on AEP. Re-run
this any time you upgrade the detection libraries — an upstream regex change could shift results.

In [ ]:
cases = {
    "Googlebot": "Mozilla/5.0 (compatible; Googlebot/2.1; +http://www.google.com/bot.html)",
    "GPTBot (training)": "Mozilla/5.0 (compatible; GPTBot/1.0; +https://openai.com/gptbot)",
    "ChatGPT-User (on-demand)": "Mozilla/5.0 (compatible; ChatGPT-User/1.0; +https://openai.com/bot)",
    "PerplexityBot (search)": "Mozilla/5.0 (compatible; PerplexityBot/1.0; +https://perplexity.ai/perplexitybot)",
    "python-requests": "python-requests/2.31.0",
    "curl": "curl/8.4.0",
    "HeadlessChrome": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 HeadlessChrome/120.0.0.0 Safari/537.36",
    "Chrome desktop (human)": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "iPhone Safari (human)": "Mozilla/5.0 (iPhone; CPU iPhone OS 17_1 like Mac OS X) AppleWebKit/605.1.15 Version/17.1 Mobile/15E148 Safari/604.1",
    "CUBOT (known false positive)": "Mozilla/5.0 (Linux; Android 11; CUBOT NOTE 20 PRO) AppleWebKit/537.36 Chrome/95.0 Mobile Safari/537.36",
}

classify(list(cases.values()), detectors, MIN_VOTES, categorizer).assign(case=list(cases.keys()))[
    ["case", "User_agent", "is_bot", "votes", "detected_by", "category"]
]

## 4. The query

One query instead of one-per-dataset: an anti-join against your bot table, done server-side in AEP.
Only user-agents that were NEVER seen before come back to Python.

**Note on field paths**: every field below is built from `XDM_NAMESPACE` (imported from
`bot_detection.py`). If a field path is wrong for your schema, this is the cell to check first —
use `SHOW COLUMNS IN <dataset>` in the AEP Query Service UI to see your real field paths.

In [ ]:
print(build_union_query(DATASETS, DAYS, BOT_TABLE).as_string(conn))

### Run it

In [ ]:
with conn.cursor() as cur:
    print(f"Querying across {len(DATASETS)} datasets...")
    cur.execute(build_union_query(DATASETS, DAYS, BOT_TABLE))
    user_agents = [row[0] for row in cur if row[0]]

print(f"New user-agents to classify: {len(user_agents):,}")
user_agents[:5]

## 5. Classification

In [ ]:
classified = classify(user_agents, detectors, MIN_VOTES, categorizer)

print(f"Bots detected: {classified['is_bot'].sum():,} out of {len(classified):,} "
      f"({100 * classified['is_bot'].mean():.1f}%)")

categorized = classified.loc[classified["is_bot"] & (classified["category"] != "")]
if not categorized.empty:
    print("Categories among detected bots:")
    print(categorized["category"].value_counts().to_string())

classified.head(10)

### Optional: parallel classification

Only worth it above roughly 20-30k unique user-agents, and only with multiple CPU cores available —
see the comment above `classify_parallel` in `bot_detection.py` for the reasoning and the measured
fixed cost per worker. Uncomment to use it instead of the cell above.

In [ ]:
# classified = classify_parallel(user_agents, MIN_VOTES, n_workers=4)

## 6. Review

### 6a. Who found what

Breakdown of detected bots by which detector(s) voted for them. A bot confirmed by all libraries at
once is a strong signal; a single-detector catch is where false positives tend to hide.

In [ ]:
(classified[classified["is_bot"]]
    .groupby("detected_by")
    .size()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("n_user_agents"))

### 6b. Disputed cases

User-agents where detectors disagree (some voted bot, some didn't). This is where false positives —
and false negatives — tend to concentrate.

In [ ]:
n_detectors = len(detectors)
disputed = classified[(classified["votes"] > 0) & (classified["votes"] < n_detectors)]
print(f"{len(disputed):,} user-agents without unanimous agreement")
disputed.sort_values("votes").head(30)

### 6c. What was NOT flagged

Sample the non-bots: if anything looks off (exotic HTTP clients, malformed UAs, machine-generated
strings), it's material for expanding `BOT_PATTERN` in `bot_detection.py`.

In [ ]:
classified[~classified["is_bot"]].sample(min(30, (~classified["is_bot"]).sum()), random_state=0)

### 6d. Categories among detected bots

How many are "good" AI crawlers (worth understanding, not blocking blindly) versus classic search
bots, monitors, or generic scrapers.

In [ ]:
(classified[classified["is_bot"] & (classified["category"] != "")]
    .groupby("category")
    .size()
    .sort_values(ascending=False)
    .to_frame("n_user_agents"))

## 7. The delta to upload

Three columns: user-agent, bot flag, and category. The category column requires that field to
already exist in your `BOT_TABLE`'s XDM schema — if it's named differently there, rename it before
uploading (see `UPLOAD_COLUMNS` in `bot_detection.py`).

In [ ]:
df_final = classified.loc[classified["is_bot"], ["User_agent", "category"]].copy()
df_final["BOT_DETECTION"] = 1
df_final = df_final[["User_agent", "BOT_DETECTION", "category"]].sort_values("User_agent").reset_index(drop=True)

classified.to_csv("audit_bot_detection.csv", index=False, encoding="utf-8")

print(f"Rows to upload: {len(df_final):,}")
if len(df_final) > MAX_EXPECTED_ROWS:
    print(f"WARNING: above the expected threshold ({MAX_EXPECTED_ROWS:,}). Review before uploading.")
if df_final.empty:
    print("No new bots: skipping upload (an empty CSV could break the downstream data source).")

df_final.head()

## 8. Upload over SFTP

Last cell, deliberately separate: nothing above this touches anything remote.

Atomic write (`.tmp` + rename): downstream ingestion can never read a half-written CSV.

**First time**, register the host key. Two ways, only needed once:

- from a terminal: `ssh-keyscan -p 22 <your-sftp-host> >> ~/.ssh/known_hosts`
- or run the optional cell below, which does the same thing without leaving the notebook

In [ ]:
# RUN ONCE (skip on later runs). Registers the SFTP host key explicitly, the
# way "ssh-keyscan" would, so the upload cell below can verify it with RejectPolicy.
import paramiko  # SftpConfig gia' importato nella prima cella, da bot_detection.aep_pipeline

SFTP = {
    "host": os.getenv("SFTP_HOST") or input("SFTP host: "),
    "port": int(os.getenv("SFTP_PORT", "22")),
    "user": os.getenv("SFTP_USER") or input("SFTP user: "),
    "password": os.getenv("SFTP_PASSWORD") or getpass.getpass("SFTP password: "),
    "remote_path": os.getenv("SFTP_REMOTE_PATH", "bot_detection/bot.csv"),
}

known_hosts_path = os.path.expanduser("~/.ssh/known_hosts")
os.makedirs(os.path.dirname(known_hosts_path), exist_ok=True)

_probe = paramiko.SSHClient()
_probe.set_missing_host_key_policy(paramiko.AutoAddPolicy())  # only to READ the offered key
_probe.connect(
    hostname=SFTP["host"], port=SFTP["port"],
    username=SFTP["user"], password=SFTP["password"],
    look_for_keys=False, allow_agent=False, timeout=30,
)
server_key = _probe.get_transport().get_remote_server_key()
_probe.close()

from paramiko.hostkeys import HostKeys
hk = HostKeys()
if os.path.exists(known_hosts_path):
    hk.load(known_hosts_path)
hk.add(SFTP["host"], server_key.get_name(), server_key)
hk.save(known_hosts_path)

print(f"Host key for {SFTP['host']} saved to {known_hosts_path}.")

In [ ]:
assert not df_final.empty, "Nothing to upload: stop here."

import io

payload = df_final.to_csv(index=False, encoding="utf-8").encode("utf-8")
remote = SFTP["remote_path"]
tmp = remote + ".tmp"

client = paramiko.SSHClient()
client.load_system_host_keys()
client.set_missing_host_key_policy(paramiko.RejectPolicy())  # AutoAddPolicy() for testing only

try:
    client.connect(
        hostname=SFTP["host"], port=SFTP["port"],
        username=SFTP["user"], password=SFTP["password"],
        look_for_keys=False, allow_agent=False, timeout=30,
    )
    sftp = client.open_sftp()
    try:
        sftp.putfo(io.BytesIO(payload), tmp, confirm=True)
        try:
            sftp.posix_rename(tmp, remote)
        except (IOError, AttributeError):
            try:
                sftp.remove(remote)
            except IOError:
                pass
            sftp.rename(tmp, remote)
        print(f"Upload OK: {remote} — {len(df_final):,} rows, {len(payload):,} bytes")
    finally:
        sftp.close()
finally:
    client.close()

---

## Next steps

This notebook covers "honest" bots — the ones that declare themselves in the User-Agent. Bots that
mimic a real browser's UA slip past this entirely; see `behavioral_detection.ipynb` for the
session-level approach that targets exactly that gap.